# Gemini API: Chain of thought prompting

Using chain of thought helps the LLM take a logical and arithmetic approach. Instead of outputting the answer immediately, the LLM uses smaller and easier steps to get to the answer.


## Setup

### Install the Google GenAI SDK

Install the Google GenAI SDK from [npm](https://www.npmjs.com/package/@google/genai). 

```bash
$ npm install @google/genai
```

### Setup your API key

You can [create](https://aistudio.google.com/app/apikey) your API key using Google AI Studio with a single click.

Remember to treat your API key like a password. Don't accidentally save it in a notebook or source file you later commit to GitHub. In this notebook we will be storing the API key in a `.env` file. You can also set it as an environment variable or use a secret manager. 

Here's how to set it up in a `.env` file:

```bash
$ touch .env
$ echo "GEMINI_API_KEY=<YOUR_API_KEY>" >> .env
```

:::{.callout-tip}

Another option is to set the API key as an environment variable. You can do this in your terminal with the following command:

```bash
$ export GEMINI_API_KEY="<YOUR_API_KEY>"
```
:::

### Load the API key

To load the API key from the `.env` file, we will use the `dotenv` package. This package loads environment variables from a `.env` file into `process.env`. 

```bash
$ npm install dotenv
```

Then, we can load the API key in our code:


In [2]:
const dotenv = require("dotenv") as typeof import("dotenv");

dotenv.config({
  path: "../../.env",
});

const GEMINI_API_KEY = process.env.GEMINI_API_KEY ?? "";
if (!GEMINI_API_KEY) {
  throw new Error("GEMINI_API_KEY is not set in the environment variables");
}
console.log("GEMINI_API_KEY is set in the environment variables");


GEMINI_API_KEY is set in the environment variables


:::{.callout-note}
In our particular case the `.env` is is two directories up from the notebook, hence we need to use `../../` to go up two directories. If the `.env` file is in the same directory as the notebook, you can omit it altogether.

```
│
├── .env
└── examples
    └── prompting
        └── Chain_of_thought_prompting.ipynb
```
:::


### Initialize SDK Client

With the new SDK, now you only need to initialize a client with you API key (or OAuth if using [Vertex AI](https://cloud.google.com/vertex-ai)). The model is now set in each call.


In [3]:
const google = require("@google/genai") as typeof import("@google/genai");

const ai = new google.GoogleGenAI({ apiKey: GEMINI_API_KEY });


### Select a model

Now select the model you want to use in this guide, either by selecting one in the list or writing it down. Keep in mind that some models, like the 2.5 ones are thinking models and thus take slightly more time to respond (cf. [thinking notebook](../../quickstarts/Get_started_thinking.ipynb) for more details and in particular learn how to switch the thiking off).


In [4]:
const tslab = require("tslab") as typeof import("tslab");

const MODEL_ID = "gemini-2.5-flash-preview-05-20";


## Example

Sometimes LLMs can return non-satisfactory answers. To simulate that behavior, you can implement a phrase like "Return the answer immediately" in your prompt.

Without this, the model sometimes uses chain of thought by itself, but it is inconsistent and does not always result in the correct answer.


In [5]:
const immediate_response = await ai.models.generateContent({
  model: MODEL_ID,
  contents:
    "5 people can create 5 donuts every 5 minutes. How much time would it take 25 people to make 100 donuts? Return the answer immediately.",
});
tslab.display.markdown(immediate_response.text ?? "");


It would take 20 minutes.

To influence this you can implement chain of thought into your prompt and look at the difference in the response. Note the multiple steps within the prompt.


In [ ]:
const cot_prompt = `
  Question: 11 factories can make 22 cars per hour. How much time would it take 22 factories to make 88 cars?
  Answer: A factory can make 22/11=2 cars per hour. 22 factories can make 22*2=44 cars per hour. Making 88 cars would take 88/44=2 hours. The answer is 2 hours.
  Question: 5 people can create 5 donuts every 5 minutes. How much time would it take 25 people to make 100 donuts?
  Answer:
`;

const cot_response = await ai.models.generateContent({
  model: MODEL_ID,
  contents: cot_prompt,
});
tslab.display.markdown(cot_response.text ?? "");


A person can create 5/5 = 1 donut every 5 minutes. 25 people can create 25\*1 = 25 donuts every 5 minutes. Making 100 donuts would take (100/25) \* 5 = 4 \* 5 = 20 minutes. The answer is 20 minutes.

## Next steps

Be sure to explore other examples of prompting in the repository. Try writing prompts about classifying your own data, or try some of the other prompting techniques such as few-shot prompting.
